# Commercialization Intelligence Engine

**Pipeline:** Customer signals -> ML pattern detection -> AI explanation -> Commercial decision

This notebook presents the ranked portfolio, model outputs, SHAP explainability, and executive summary.

> **Before running:** Ensure Phase 1-2 outputs exist. If not, run:
> ```bash
> python data/generate_mock_data.py
> python data/prepare_features.py
> ```
>
> **Interactive dashboard:** run `streamlit run app/streamlit_app.py` from the project root.
>
> **To run this notebook:** Kernel -> Restart & Run All

## Problem Approach

### Assumptions
- Customer behavior data is synthetic, generated with a hidden `_latent_commercial_potential` variable (Beta(2,2) distributed) that drives all downstream signals.
- Real-world outcome labels are unavailable, so pseudo-labels are assigned via rule-based commercial judgment logic to train the classifier.
- Each concept is evaluated independently; portfolio-level decisions are derived by ranking readiness scores.

### Data Design
- **12 product concepts** across 7 industries. Metadata follows a dependency chain: Industry -> Problem Area -> Target User -> Concept Name.
- **80 customers** across 5 segments (Enterprise, Mid-Market, SMB, Public Sector, Startup).
- **5 data areas:** product concepts, customer demo signals, sandbox usage, commercial signals, and text feedback.
- Missing values (~4%), outliers (~3%), and noise are intentionally injected to test the cleaning pipeline.

### ML Method Choice
- **Baseline weighted score** (deterministic, interpretable) provides a readable ranking (1-100).
- **K-Means clustering** (k=4) identifies natural groupings by demand intensity and feasibility risk.
- **Random Forest classifier** (200 trees, max_depth=4) predicts one of 5 commercialization outcomes. Chosen for: (a) robust on small datasets, (b) native feature importance, (c) compatible with SHAP TreeExplainer.
- **SHAP values** provide per-prediction feature attribution for explainability.
- Text feedback features (sentiment, objection count, capability requests) are extracted and included in the ML pipeline.

### Decision Logic
- Final readiness = 70% baseline weighted score + 30% ML classifier confidence.
- Confidence = 55% data-volume confidence + 45% model certainty.
- Five outcomes: MVP Build, Customer Pilot, Reusable Asset, Incubate, Archive.

### Limitations
- All data is synthetic; classifier is trained on rule-based pseudo-labels, not real-world outcomes.
- With 12 concepts, cross-validation is illustrative, not statistically rigorous.
- Text feedback uses keyword-based sentiment, not transformer embeddings.

## 1. Setup & Run Pipeline

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from models.decision_engine import run_decision_engine
from models.insight_layer import build_insights, generate_executive_summary, FEATURE_LABELS

PROCESSED = ROOT / "data" / "processed"
FEATURES_PATH = PROCESSED / "concept_features.csv"

if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Run Phase 1-2 first:\n"
        "  python data/generate_mock_data.py\n"
        "  python data/prepare_features.py"
    )

report, artifacts = run_decision_engine(FEATURES_PATH, PROCESSED)
insights = build_insights(artifacts)
ranked = insights.sort_values("portfolio_rank")

print(f"Concepts analyzed: {len(ranked)}")
print(f"Top concept: {ranked.iloc[0]['concept_name']} ({ranked.iloc[0]['readiness_score']}/100)")

## 2. Data Validation & Quality Report

In [ ]:
with open(PROCESSED / "validation_report.json") as f:
    val_report = json.load(f)

print(f"Validation passed: {val_report['validation_passed']}")
print(f"Concepts: {val_report['concept_count']}")
print(f"Interactions: {val_report['interaction_count']}")

if val_report["validation_issues"]:
    print("\nIssues:")
    for issue in val_report["validation_issues"]:
        print(f"  - {issue}")

print("\nFeature statistics (concept-level):")
feat_df = pd.DataFrame(val_report["feature_summary"]).T
feat_df

## 3. Executive Summary

In [ ]:
summary = generate_executive_summary(artifacts["ranked"])
print(summary)

## 4. Ranked Portfolio Recommendations

In [ ]:
display_cols = [
    "portfolio_rank", "concept_name", "industry", "problem_area",
    "readiness_score", "confidence_score", "recommended_outcome",
]
ranked[display_cols]

## 5. Visual Analytics

In [ ]:
OUTCOME_COLORS = {
    "MVP Build": "#0e9f6e",
    "Customer Pilot": "#1f6feb",
    "Reusable Asset": "#8957e5",
    "Incubate": "#bf8700",
    "Archive": "#cf222e",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Readiness ranking
ordered = ranked.sort_values("readiness_score", ascending=True)
colors = [OUTCOME_COLORS.get(o, "#95a5a6") for o in ordered["recommended_outcome"]]
axes[0, 0].barh(ordered["concept_name"], ordered["readiness_score"], color=colors, height=0.55)
axes[0, 0].set_xlabel("Readiness Score (1-100)")
axes[0, 0].set_title("Commercialization Readiness by Concept")
axes[0, 0].set_xlim(0, 100)

# Outcome distribution
counts = ranked["recommended_outcome"].value_counts()
pie_colors = [OUTCOME_COLORS.get(o, "#95a5a6") for o in counts.index]
axes[0, 1].pie(counts, labels=counts.index, autopct="%1.0f%%", colors=pie_colors, startangle=140)
axes[0, 1].set_title("Recommendation Distribution")

# K-Means clusters
df = artifacts["df"]
cluster_colors = ["#1f6feb", "#0e9f6e", "#bf8700", "#cf222e", "#8957e5", "#e16f24"]
for cid, group in df.groupby("cluster_id"):
    color = cluster_colors[cid % len(cluster_colors)]
    axes[1, 0].scatter(
        group["demand_intensity"], group["feasibility_risk"],
        s=group["readiness_score"] * 3.5, alpha=0.7,
        label=group["cluster_profile"].iloc[0], color=color,
        edgecolors="white", linewidth=0.8,
    )
axes[1, 0].set_xlabel("Demand Intensity")
axes[1, 0].set_ylabel("Feasibility Risk")
axes[1, 0].set_title("Cluster Analysis: Demand vs Delivery Effort")
axes[1, 0].legend(fontsize=7, loc="best")

# Feature importance
importance = artifacts["importance"].head(8)
importance_sorted = importance.sort_values("importance", ascending=True)
labels = [FEATURE_LABELS.get(f, f) for f in importance_sorted["feature"]]
bars = axes[1, 1].barh(labels, importance_sorted["importance"], color=plt.cm.Blues_r([0.4, 0.5, 0.6, 0.7, 0.8]), height=0.5)
axes[1, 1].set_xlabel("Relative Importance")
axes[1, 1].set_title("Feature Importance (Random Forest)")

plt.tight_layout()
plt.show()

## 6. SHAP Explainability - Why Each Decision?

In [ ]:
feature_names = artifacts["feature_columns"]

for _, row in ranked.iterrows():
    shap_cols = [f"shap_{f}" for f in feature_names if f"shap_{f}" in row.index]
    shap_vals = [(c.replace("shap_", ""), row[c]) for c in shap_cols]
    shap_vals.sort(key=lambda x: abs(x[1]), reverse=True)
    top = shap_vals[:4]

    print(f"\n{'='*70}")
    print(f"#{int(row['portfolio_rank'])} {row['concept_name']} -> {row['recommended_outcome']}")
    print(f"Readiness: {row['readiness_score']}/100 | Confidence: {row['confidence_score']:.0%}")
    print(f"\nAI Narrative: {row['ai_narrative']}")
    print("\nTop SHAP drivers:")
    for feat, val in top:
        label = FEATURE_LABELS.get(feat, feat)
        direction = "(+) toward outcome" if val > 0 else "(-) against outcome"
        print(f"  * {label}: {val:+.4f} {direction}")

## 7. SHAP Waterfall - Selected Concept

In [ ]:
CONCEPT = ranked.iloc[0]["concept_name"]  # change to explore others
row = ranked[ranked["concept_name"] == CONCEPT].iloc[0]

shap_cols = [f"shap_{f}" for f in feature_names if f"shap_{f}" in row.index]
labels = [FEATURE_LABELS.get(c.replace("shap_", ""), c) for c in shap_cols]
values = [row[c] for c in shap_cols]
pairs = sorted(zip(labels, values), key=lambda x: abs(x[1]), reverse=True)[:8]
labels, values = zip(*pairs)
bar_colors = ["#0e9f6e" if v > 0 else "#cf222e" for v in values]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(labels, values, color=bar_colors, height=0.45)
ax.axvline(0, color="#8b949e", linewidth=0.8, linestyle="--")
ax.set_xlabel("SHAP contribution")
ax.set_title(f"Feature Contributions: {row['concept_name']} -> {row['recommended_outcome']}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Cluster Summary

In [ ]:
artifacts["cluster_summary"]

## 9. Model Report & Cross-Validation

In [ ]:
with open(PROCESSED / "model_report.json") as f:
    model_report = json.load(f)

print("Baseline weights:")
for k, v in model_report["baseline_model"]["weights"].items():
    print(f"  {k}: {v}")

print("\nOutcome distribution:")
for outcome, count in model_report["classifier"]["outcome_distribution"].items():
    print(f"  {outcome}: {count}")

print("\nTop features:")
for feat in model_report["top_features"]:
    print(f"  {feat['feature']}: {feat['importance']:.3f}")

cv = model_report.get("cross_validation", {})
if cv:
    print(f"\nCross-validation ({cv.get('method', 'N/A')}):")
    print(f"  Mean accuracy: {cv.get('accuracy_mean', 'N/A'):.2%}")
    print(f"  Std: +/- {cv.get('accuracy_std', 'N/A'):.2%}")
    print(f"  Fold scores: {cv.get('fold_scores', [])}")

## 10. Streamlit Dashboard

For interactive exploration (concept drill-down, live filters), launch the Streamlit app from the project root:

```bash
streamlit run app/streamlit_app.py
```